# Memory Optimization and Efficient Data Processing


## Memory Optimization 

Memory optimization is crucial when working with large datasets that may not fit into RAM. Techniques include reducing data types, using efficient storage formats, and processing data in chunks.

### Why Optimize Memory?
- Prevent Out-of-Memory (OOM) errors.
- Speed up computations by reducing data size.
- Enable handling of larger datasets on limited hardware.

### Techniques
1. **Data Type Downcasting**: Convert to smaller data types (e.g., float64 to float32).
2. **Categorical Data**: Use categories for repetitive strings.
3. **Chunking**: Process data in smaller pieces.
4. **Efficient File Formats**: Use Parquet instead of CSV.

In [9]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sys
import dask.dataframe as dd
from scipy import sparse

In [10]:
# Function to check memory usage
def memory_usage(df):
    return df.memory_usage(deep=True).sum() / (1024 ** 2)  # in MB

In [11]:
# Create a sample large dataset
np.random.seed(42)
data = {
    'A': np.random.randn(10000000),
    'B': np.random.randint(0, 100, 10000000),
    'C': np.random.choice(['cat1', 'cat2', 'cat3', 'cat4'], 10000000)
}
df = pd.DataFrame(data)
print(f"Original Memory Usage: {memory_usage(df):.2f} MB")

Original Memory Usage: 658.04 MB


### 1. Data Type Downcasting
Downcasting is the process of converting a column in a DataFrame to a smaller, more memory-efficient data type without losing information. This reduces memory usage, which is crucial when handling large datasets.

#### Downcasting helps by : 
- Pandas often defaults numeric columns to int64 or float64.
- Many datasets don’t need such large types; smaller types (int8, int16, float32) can save memory.
- Example: A column with numbers 0–100 can easily fit in int8 instead of int64.

Pandas provides `pd.to_numeric()` and `astype()` for downcasting. 

In [12]:
# Downcast numerical columns using astype
def downcast_df(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df

df_opt = downcast_df(df.copy())
print(f"After Downcasting: {memory_usage(df_opt):.2f} MB")

After Downcasting: 581.74 MB


In [13]:
df_opt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 3 columns):
 #   Column  Dtype  
---  ------  -----  
 0   A       float32
 1   B       int32  
 2   C       object 
dtypes: float32(1), int32(1), object(1)
memory usage: 152.6+ MB


### 2. Using Categorical Data
Pandas usually stores text/string columns as object dtype, which uses a lot of memory, especially for repeated values.

Converting a column to category replaces the strings with integer codes, dramatically reducing memory usage.

In [14]:
# Convert string column to category
df_opt['C'] = df_opt['C'].astype('category')
print(f"After Categorical Conversion: {memory_usage(df_opt):.2f} MB")

After Categorical Conversion: 85.83 MB


In [15]:
df_opt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 3 columns):
 #   Column  Dtype   
---  ------  -----   
 0   A       float32 
 1   B       int32   
 2   C       category
dtypes: category(1), float32(1), int32(1)
memory usage: 85.8 MB


### 3. Processing in Chunks

Processing in chunks (also called batch processing) is the technique of reading or processing data in smaller, manageable pieces rather than loading the entire dataset into memory at once.

In [16]:
# Simulate large CSV file
df.to_csv('large_data.csv', index=False)

# Read in chunks
chunks = pd.read_csv('large_data.csv', chunksize=1000)
processed_chunks = []
for chunk in chunks:
    # Process each chunk (e.g., sum column A)
    processed_chunks.append(chunk['A'].sum())

total_sum = sum(processed_chunks)
print(f"Total Sum of Column A: {total_sum}")

Total Sum of Column A: -639.5751574849666


### 4. Efficient File Formats (Parquet)

Parquet is a highly efficient, columnar storage file format designed to optimize both disk space and memory usage, especially for large datasets.

Parquet also uses compression and encoding techniques to efficiently store repeated or sparse values. It supports chunked reading, so large datasets can be processed in manageable batches without loading everything into memory.

These features make Parquet ideal for big data analytics, machine learning, and ETL tasks, providing fast and memory-efficient data handling.

Parquet reduces disk size, not memory usage when fully loaded but reducing memory while loading a specific column.

In [18]:
# Save as Parquet
df.to_parquet('large_data.parquet')

# Load Parquet
df_parquet = pd.read_parquet('large_data.parquet')
print(f"Parquet Memory Usage: {memory_usage(df_parquet):.2f} MB")

Parquet Memory Usage: 658.04 MB


## Efficient Data Processing 

Efficient data processing involves techniques to speed up computations, such as vectorization, parallel processing, and optimized operations.

### Why Efficient Processing?
- Reduce computation time.
- Handle big data scalably.
- Improve overall workflow productivity.

### Techniques
1. **Vectorization**: Use array operations instead of loops.
2. **NumPy for Numerical Computations**: Faster than pure Python.
3. **Parallel Processing with Dask**: For large-scale data.

### 1. Vectorization vs Loops

Vectorization means performing operations on entire arrays or columns at once rather than iterating element by element.

Achieved using libraries like NumPy, Pandas, or frameworks like TensorFlow and PyTorch.

Utilizes low-level C/Fortran implementations for speed and memory efficiency.

In [19]:
import time

# Loop version
start = time.time()
result_loop = []
for i in range(len(df['A'])):
    result_loop.append(df['A'][i] * 2)
print(f"Loop Time: {time.time() - start:.4f} seconds")

# Vectorized
start = time.time()
result_vec = df['A'] * 2
print(f"Vectorized Time: {time.time() - start:.4f} seconds")

Loop Time: 44.3041 seconds
Vectorized Time: 0.0349 seconds


### 2. Using NumPy 

NumPy (Numerical Python) is a library for numerical computing in Python.

Provides high-performance arrays (ndarrays) and vectorized operations.

Uses contiguous memory blocks and C/Fortran backend, which makes it faster and more memory-efficient than Python lists.

In [20]:
# NumPy array operations
arr = np.array(df['A'])
start = time.time()
np_result = np.sin(arr) + np.cos(arr)
print(f"NumPy Operation Time: {time.time() - start:.4f} seconds")

NumPy Operation Time: 0.4852 seconds


### 3. Parallel Processing with Dask 
Dask allows Python programs to process large datasets efficiently by splitting the data into smaller chunks and performing computations in parallel across multiple CPU cores, which reduces memory usage and speeds up data processing compared to traditional single-core methods.

In [21]:
# Use Dask for large data
ddf = dd.from_pandas(df, npartitions=4)
start = time.time()
mean_dask = ddf['A'].mean().compute()
print(f"Dask Mean: {mean_dask}, Time: {time.time() - start:.4f} seconds")

# Pandas equivalent
start = time.time()
mean_pd = df['A'].mean()
print(f"Pandas Mean: {mean_pd}, Time: {time.time() - start:.4f} seconds")

Dask Mean: -6.39575157484955e-05, Time: 3.9671 seconds
Pandas Mean: -6.39575157484955e-05, Time: 0.0209 seconds
